# Daily LightGBM checks of the replan on Kaggle CPU

Three sets of runs of `python -m earlymule.models.lightgbm_export` (full feature set) on the main dataset, CPU only so they do not use the weekly GPU quota:

- **1% of labelled cases, more draws**: seeds 3 to 9 next to seeds 0 to 2 of the label-scarcity runs, because at 1% only 2 cases keep their labels and which ones decides the result.
- **Shuffled labels**: the training labels are permuted (`--shuffle-labels`); a pipeline check, expected near random ranking.
- **Unseen typology**: every training row of the mules of one typology is removed (`--leave-out-typology`), five typologies by three seeds; scored on that typology's test mules against the full-label run `lgbm_abl_full`.

Inputs: the private datasets `verzol/earlymule-80k-earlymule-mule` (export) and `verzol/earlymule-80k-label-cases` (case map with typology, `python -m earlymule.data.label_fraction`). Each run writes scores and `run_meta.json` to `/kaggle/working/<tag>/seed<N>/`; evaluation runs locally with `python -m earlymule.evaluation.import_scores`. One notebook per batch is generated by `notebooks/kaggle/lgbm_checks/make_batches.py`, which rewrites the parameter cell.


In [ ]:
# Run parameters: dataset, (output tag, extra arguments, seed) jobs run one after another ({cases} is the case map); commit to check out
DATASET = '80K-earlymule-mule'
JOBS = [('lgbm_shuffled', ['--shuffle-labels'], 0)]
COMMIT = None

In [ ]:
# Clone the repository at COMMIT and install it without dependencies (LightGBM, pandas and PyYAML are preinstalled)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!nproc; free -g

In [ ]:
# Train every job in order; a failed run is logged and the next one still starts
import json, subprocess, time
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == DATASET)
cases = next(Path('/kaggle/input').rglob(f'label_cases_{DATASET}.csv'))
codes = []
for tag, extra, seed in JOBS:
    out = Path('/kaggle/working') / tag / f'seed{seed}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    cmd = ['python', '-m', 'earlymule.models.lightgbm_export', '--data', str(data), '--ablation', 'full',
           '--seed', str(seed), '--out', str(out)] + [a.format(cases=cases) for a in extra]
    code = subprocess.call(cmd, cwd='/tmp/earlymule-tg', stdout=open(out / 'run.log', 'w'), stderr=subprocess.STDOUT)
    codes.append((tag, seed, code, round((time.time() - t0) / 60, 1)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last lines of every run log
for log in sorted(Path('/kaggle/working').glob('lgbm_*/seed*/run.log')):
    print(f'--- {log.parent.parent.name}/{log.parent.name}')
    print(log.read_text()[-1500:])